# Animal Detection MoE — Colab Training

**Before running:** `Runtime → Change runtime type → T4 GPU`

This notebook is self-contained — run all cells top to bottom.

**Disconnect-safe:** Cell 3 mounts your Google Drive and symlinks `saved_models/` directly to it.
Every checkpoint is written to Drive in real-time, so a runtime crash or disconnect loses nothing.

In [ ]:
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU found! Go to Runtime > Change runtime type > T4 GPU'

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────────
# Models and checkpoints are saved DIRECTLY to Drive in real-time.
# A runtime disconnect will NOT lose any completed checkpoint.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/MoE_Animal_Detection'
os.makedirs(f'{DRIVE_DIR}/saved_models', exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/plots', exist_ok=True)
print(f'Drive folder ready: {DRIVE_DIR}')

In [ ]:
# ── Step 2: Create local code dirs + symlink outputs to Drive ───────────────
# Code files live in fast local Colab storage.
# saved_models/ and plots/ are symlinked to Drive so writes go there directly.
import os

DRIVE_DIR = '/content/drive/MyDrive/MoE_Animal_Detection'

for d in ['models', 'experts', 'gating', 'utils']:
    os.makedirs(d, exist_ok=True)

for local, remote in [
    ('saved_models', f'{DRIVE_DIR}/saved_models'),
    ('plots',        f'{DRIVE_DIR}/plots'),
]:
    if os.path.islink(local):
        print(f'{local} already symlinked.')
    elif os.path.exists(local):
        print(f'{local} exists as a real dir — skipping symlink.')
    else:
        os.symlink(remote, local)
        print(f'Linked: {local} -> {remote}')

print('All directories ready.')

In [ ]:
%%writefile utils/data_loader.py
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split

ARTIFICIAL_CLASSES = [0, 1, 8, 9]
NATURAL_CLASSES = [2, 3, 4, 5, 6, 7]

def load_and_preprocess_cifar10():
    (x_train_full, y_train_full), (x_test, y_test) = cifar10.load_data()
    x_train_full = x_train_full.astype('float32') / 255.0
    x_test = x_test.astype('float32') / 255.0
    x_train, x_val, y_train, y_val = train_test_split(
        x_train_full, y_train_full, test_size=0.1, random_state=42, stratify=y_train_full
    )
    return (x_train, y_train), (x_val, y_val), (x_test, y_test)

def get_expert_subsets(x, y):
    y_flat = y.flatten()
    art_mask = np.isin(y_flat, ARTIFICIAL_CLASSES)
    nat_mask = np.isin(y_flat, NATURAL_CLASSES)
    return (x[art_mask], y[art_mask]), (x[nat_mask], y[nat_mask])

def get_gating_labels(y):
    y_flat = y.flatten()
    gate_labels = np.zeros_like(y_flat)
    gate_labels[np.isin(y_flat, NATURAL_CLASSES)] = 1
    return gate_labels.reshape(-1, 1)

def to_one_hot(y, num_classes=10):
    return to_categorical(y, num_classes=num_classes)

def get_augmented_generator(x, y, batch_size=64):
    datagen = ImageDataGenerator(
        horizontal_flip=True,
        width_shift_range=0.125,
        height_shift_range=0.125,
        fill_mode='reflect'
    )
    return datagen.flow(x, y, batch_size=batch_size)

In [ ]:
%%writefile utils/visualization.py
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np
import os

def plot_training_history(history, title='Training History'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(history.history['accuracy'], label='Train Accuracy')
    if 'val_accuracy' in history.history:
        ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax1.set_title(f'{title} - Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Accuracy')
    ax1.legend()
    ax2.plot(history.history['loss'], label='Train Loss')
    if 'val_loss' in history.history:
        ax2.plot(history.history['val_loss'], label='Validation Loss')
    ax2.set_title(f'{title} - Loss')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss')
    ax2.legend()
    plt.tight_layout()
    os.makedirs('plots', exist_ok=True)
    plt.savefig(f'plots/{title}_history.png')
    plt.close()

def plot_confusion_matrix(y_true, y_pred, classes, title='Confusion Matrix'):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(title)
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

In [ ]:
%%writefile models/base_cnn.py
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.regularizers import l2

def create_base_cnn(input_shape=(32, 32, 3), num_classes=10):
    reg = l2(1e-4)
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=reg, input_shape=input_shape),
        BatchNormalization(),
        Conv2D(32, (3, 3), activation='relu', padding='same', kernel_regularizer=reg),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=reg),
        BatchNormalization(),
        Conv2D(64, (3, 3), activation='relu', padding='same', kernel_regularizer=reg),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        Conv2D(128, (3, 3), activation='relu', padding='same', kernel_regularizer=reg),
        BatchNormalization(),
        Conv2D(128, (3, 3), activation='relu', padding='same', kernel_regularizer=reg),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Dropout(0.25),
        GlobalAveragePooling2D(),
        Dense(256, activation='relu', kernel_regularizer=reg),
        BatchNormalization(),
        Dropout(0.5),
        Dense(num_classes, activation='softmax' if num_classes > 1 else 'sigmoid')
    ])
    return model

In [ ]:
%%writefile models/moe_model.py
import os
import numpy as np
import tensorflow as tf

class HierarchicalMoE:
    def __init__(self, models_dir='saved_models'):
        self.models_dir = models_dir
        print('Loading experts...')
        self.base_expert = tf.keras.models.load_model(os.path.join(models_dir, 'base_expert_final.keras'))
        self.art_expert  = tf.keras.models.load_model(os.path.join(models_dir, 'artificial_expert_final.keras'))
        self.nat_expert  = tf.keras.models.load_model(os.path.join(models_dir, 'natural_expert_final.keras'))
        print('Loading gating networks...')
        self.first_level_gate = tf.keras.models.load_model(os.path.join(models_dir, 'first_level_gate_final.keras'))
        self.art_gater = tf.keras.models.load_model(os.path.join(models_dir, 'artificial_gater_final.keras'))
        self.nat_gater = tf.keras.models.load_model(os.path.join(models_dir, 'natural_gater_final.keras'))
        print('All models loaded.')

    def predict(self, x, batch_size=32, verbose=1):
        p_nat = self.first_level_gate.predict(x, batch_size=batch_size, verbose=verbose)
        p_art = 1.0 - p_nat
        w_art = self.art_gater.predict(x, batch_size=batch_size, verbose=verbose)
        w_nat = self.nat_gater.predict(x, batch_size=batch_size, verbose=verbose)
        pred_base = self.base_expert.predict(x, batch_size=batch_size, verbose=verbose)
        pred_art  = self.art_expert.predict(x, batch_size=batch_size, verbose=verbose)
        pred_nat  = self.nat_expert.predict(x, batch_size=batch_size, verbose=verbose)
        branch_art = w_art[:, 0:1] * pred_base + w_art[:, 1:2] * pred_art
        branch_nat = w_nat[:, 0:1] * pred_base + w_nat[:, 1:2] * pred_nat
        return p_art * branch_art + p_nat * branch_nat

    def evaluate(self, x, y_true_oh, batch_size=32):
        from tensorflow.keras.metrics import CategoricalAccuracy
        preds = self.predict(x, batch_size=batch_size, verbose=0)
        acc = CategoricalAccuracy()
        acc.update_state(y_true_oh, preds)
        return acc.result().numpy()

In [ ]:
%%writefile experts/train_experts.py
import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from models.base_cnn import create_base_cnn
from utils.data_loader import load_and_preprocess_cifar10, get_expert_subsets, to_one_hot, get_augmented_generator
from utils.visualization import plot_training_history
import argparse

def train_model(model, x_train, y_train, x_val, y_val, model_name, batch_size=64, epochs=50):
    print(f'\n--- Training {model_name} ---')
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    os.makedirs('saved_models', exist_ok=True)
    best_path = os.path.join('saved_models', f'{model_name}_best.keras')
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
        ModelCheckpoint(best_path, monitor='val_accuracy', save_best_only=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
    ]
    train_gen = get_augmented_generator(x_train, y_train, batch_size=batch_size)
    steps_per_epoch = len(x_train) // batch_size
    history = model.fit(
        train_gen,
        steps_per_epoch=steps_per_epoch,
        validation_data=(x_val, y_val),
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )
    plot_training_history(history, title=model_name)
    final_path = os.path.join('saved_models', f'{model_name}_final.keras')
    model.save(final_path)
    print(f'Saved {model_name} -> {final_path}')
    return model

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--epochs', type=int, default=50)
    args = parser.parse_args()
    (x_train, y_train), (x_val, y_val), _ = load_and_preprocess_cifar10()
    base_model = create_base_cnn(num_classes=10)
    train_model(base_model, x_train, to_one_hot(y_train), x_val, to_one_hot(y_val), 'base_expert', epochs=args.epochs)
    (x_art_tr, y_art_tr), _ = get_expert_subsets(x_train, y_train)
    (x_art_val, y_art_val), _ = get_expert_subsets(x_val, y_val)
    art_model = create_base_cnn(num_classes=10)
    train_model(art_model, x_art_tr, to_one_hot(y_art_tr), x_art_val, to_one_hot(y_art_val), 'artificial_expert', epochs=args.epochs)
    _, (x_nat_tr, y_nat_tr) = get_expert_subsets(x_train, y_train)
    _, (x_nat_val, y_nat_val) = get_expert_subsets(x_val, y_val)
    nat_model = create_base_cnn(num_classes=10)
    train_model(nat_model, x_nat_tr, to_one_hot(y_nat_tr), x_nat_val, to_one_hot(y_nat_val), 'natural_expert', epochs=args.epochs)

if __name__ == '__main__':
    main()

In [ ]:
%%writefile gating/first_level.py
import os, sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from utils.data_loader import load_and_preprocess_cifar10, get_gating_labels
from utils.visualization import plot_training_history
import argparse

def create_gating_cnn(input_shape=(32, 32, 3)):
    model = Sequential([
        Conv2D(16, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        GlobalAveragePooling2D(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    return model

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--epochs', type=int, default=20)
    args = parser.parse_args()
    (x_train, y_train), (x_val, y_val), _ = load_and_preprocess_cifar10()
    y_gate_train = get_gating_labels(y_train)
    y_gate_val   = get_gating_labels(y_val)
    model = create_gating_cnn()
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    os.makedirs('saved_models', exist_ok=True)
    best_path = os.path.join('saved_models', 'first_level_gate_best.keras')
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ModelCheckpoint(best_path, monitor='val_accuracy', save_best_only=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
    ]
    print('\n--- Training First Level Gating Network ---')
    history = model.fit(
        x_train, y_gate_train,
        validation_data=(x_val, y_gate_val),
        epochs=args.epochs,
        batch_size=64,
        callbacks=callbacks
    )
    plot_training_history(history, title='First_Level_Gate')
    final_path = os.path.join('saved_models', 'first_level_gate_final.keras')
    model.save(final_path)
    print(f'Saved to {final_path}')

if __name__ == '__main__':
    main()

In [ ]:
%%writefile gating/second_level.py
import os, sys
import numpy as np
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.losses import CategoricalCrossentropy
from utils.data_loader import load_and_preprocess_cifar10, to_one_hot
from utils.visualization import plot_training_history
import argparse

def create_second_level_gate(input_shape=(32, 32, 3)):
    model = Sequential([
        Conv2D(16, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D((2, 2)),
        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        GlobalAveragePooling2D(),
        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(2, activation='softmax')
    ])
    return model

def generate_pseudo_labels(images, targets, base_model_path, spec_model_path, temperature=2.0):
    print(f'  Loading models for pseudo-label generation...')
    base_model = tf.keras.models.load_model(base_model_path)
    spec_model = tf.keras.models.load_model(spec_model_path)
    cce = CategoricalCrossentropy(reduction=tf.keras.losses.Reduction.NONE)
    base_preds = base_model.predict(images, batch_size=128, verbose=0)
    spec_preds = spec_model.predict(images, batch_size=128, verbose=0)
    targets_oh = to_one_hot(targets, num_classes=10)
    base_losses = cce(targets_oh, base_preds).numpy()
    spec_losses = cce(targets_oh, spec_preds).numpy()
    spec_advantage = base_losses - spec_losses
    w_spec = 1.0 / (1.0 + np.exp(-spec_advantage * temperature))
    return np.stack([1.0 - w_spec, w_spec], axis=1)

def train_gater(x_train, y_train, x_val, y_val, base_path, spec_path, model_name, epochs):
    print(f'\nGenerating pseudo-labels for {model_name}...')
    gate_y_train = generate_pseudo_labels(x_train, y_train, base_path, spec_path)
    gate_y_val   = generate_pseudo_labels(x_val,   y_val,   base_path, spec_path)
    model = create_second_level_gate()
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    os.makedirs('saved_models', exist_ok=True)
    best_path = os.path.join('saved_models', f'{model_name}_best.keras')
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ModelCheckpoint(best_path, monitor='val_accuracy', save_best_only=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
    ]
    print(f'--- Training {model_name} ---')
    history = model.fit(
        x_train, gate_y_train,
        validation_data=(x_val, gate_y_val),
        epochs=epochs,
        batch_size=64,
        callbacks=callbacks
    )
    plot_training_history(history, title=model_name)
    final_path = os.path.join('saved_models', f'{model_name}_final.keras')
    model.save(final_path)
    return model

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--epochs', type=int, default=20)
    args = parser.parse_args()
    (x_train, y_train), (x_val, y_val), _ = load_and_preprocess_cifar10()
    base_path = os.path.join('saved_models', 'base_expert_final.keras')
    art_path  = os.path.join('saved_models', 'artificial_expert_final.keras')
    nat_path  = os.path.join('saved_models', 'natural_expert_final.keras')
    if not os.path.exists(base_path):
        print('ERROR: Train experts first (run train_experts.py).')
        return
    print('=== Training Artificial Gater (full dataset) ===')
    train_gater(x_train, y_train, x_val, y_val, base_path, art_path, 'artificial_gater', args.epochs)
    print('=== Training Natural Gater (full dataset) ===')
    train_gater(x_train, y_train, x_val, y_val, base_path, nat_path, 'natural_gater', args.epochs)

if __name__ == '__main__':
    main()

## Training
The three cells below run sequentially. Each prints progress per epoch.
- **Experts**: ~25–40 min on T4
- **Gating**: ~10–15 min total

In [ ]:
!python experts/train_experts.py --epochs 50

In [ ]:
!python gating/first_level.py --epochs 20

In [ ]:
!python gating/second_level.py --epochs 20

## Evaluation

In [ ]:
import sys, numpy as np, tensorflow as tf
sys.path.insert(0, '.')
from models.moe_model import HierarchicalMoE
from utils.data_loader import load_and_preprocess_cifar10, to_one_hot

CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

_, _, (x_test, y_test) = load_and_preprocess_cifar10()
y_test_oh = to_one_hot(y_test)

# Base expert standalone
base = tf.keras.models.load_model('saved_models/base_expert_final.keras')
base_preds = base.predict(x_test, verbose=0)
base_acc = np.mean(np.argmax(base_preds, axis=1) == y_test.flatten())
print(f'Base Expert Accuracy : {base_acc*100:.2f}%')

# Full MoE
moe = HierarchicalMoE()
moe_preds = moe.predict(x_test, verbose=0)
moe_acc = np.mean(np.argmax(moe_preds, axis=1) == y_test.flatten())
print(f'MoE Accuracy         : {moe_acc*100:.2f}%')
print(f'Improvement          : {(moe_acc - base_acc)*100:+.2f}%')

In [ ]:
# Show training curves
import matplotlib.pyplot as plt, matplotlib.image as mpimg, os
plots = sorted([f for f in os.listdir('plots') if f.endswith('.png')])
fig, axes = plt.subplots(len(plots), 1, figsize=(14, 5*len(plots)))
if len(plots) == 1: axes = [axes]
for ax, fname in zip(axes, plots):
    ax.imshow(mpimg.imread(f'plots/{fname}'))
    ax.axis('off')
    ax.set_title(fname.replace('_history.png', ''))
plt.tight_layout()
plt.show()

# Your models are already safe in Google Drive at:
#   MyDrive/MoE_Animal_Detection/saved_models/
#
# This cell zips them and also downloads to your PC (optional extra copy).
import shutil, os
from google.colab import files

DRIVE_DIR = '/content/drive/MyDrive/MoE_Animal_Detection'
zip_path = '/content/saved_models'
shutil.make_archive(zip_path, 'zip', f'{DRIVE_DIR}/saved_models')
print('Zip saved to Drive. Now downloading to your PC...')
files.download(f'{zip_path}.zip')

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('saved_models', 'zip', 'saved_models')
files.download('saved_models.zip')